# LLM-as-Router on A25

**This notebook replaces the ML router with an LLM router.** we put **both experts' full JSON into the LLM's prompt** and let it
produce the final table JSON directly.

**Prompting: few-shot (in-context learning)**: the arbitration instructions plus
**N_FEWSHOT** worked examples taken from *training* tables:

**The three things we compare**
- **Single experts**: vision alone, text alone.
- **LLM router**: the LLM that merges both experts' JSON (few-shot).
- **Oracle**: an imaginary perfect per-cell chooser; the *ceiling* a router can reach.

In [ ]:
from pathlib import Path
import os, json, re, time, hashlib
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

import Levenshtein
from tqdm import tqdm
from openai import OpenAI
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = Path("data/A25/input_images")
VISION_PRED_ROOT = Path("data/outputs/vision_expert_gpt_run")
TEXT_PRED_ROOT = Path("data/outputs/text_expert_gpt_run")
DOMAINS= ["Biology", "CompSci", "ICDAR", "MatSci"]
GT_SUBDIR = "xmls"
VISION_PRED_SUBDIR = "predictions"
TEXT_PRED_SUBDIR = "nougat/predictions"

LEV_THRESHOLD  = 1.0      
RANDOM_STATE   = 27
TEST_SIZE      = 0.25     
VAL_FRACTION   = 0.20     


ROUTER_MODEL_NAME = "gpt-5.6" # previous runs: "GLM-5.2-thinking-high"

client = OpenAI(
    api_key="APIKEY",
    base_url="https://api.openai.com/v1"
)               

SLEEP_BETWEEN_CALLS_SEC = 0.0

N_FEWSHOT = 3     # in-context worked examples in the prompt
FEWSHOT_MAX_CELLS = 20    # only small train tables become examples (keeps prompts short)

MAX_RETRIES = 2
RETRY_BACKOFF_SEC = 2.0

CACHE_ROOT = Path("data/outputs/llm_router_A25/gpt") / re.sub(r"[^A-Za-z0-9._-]+", "_", ROUTER_MODEL_NAME)

CELL_SCHEMA = {
    "type": "object",
    "properties": {
        "cells": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "sr": {"type": "integer"},
                    "er": {"type": "integer"},
                    "sc": {"type": "integer"},
                    "ec": {"type": "integer"},
                    "text": {"type": "string"},
                },
                "required": ["sr", "er", "sc", "ec", "text"],
            },
        }
    },
    "required": ["cells"],
}

print("Domains:", DOMAINS)
print("Router model:", ROUTER_MODEL_NAME)

## Step 1 — Helper functions

In [ ]:

def normalize_text(text):
    if text is None:
        return ""
    
    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")

    return re.sub(r"\s+", " ", text).strip()

def lev_sim(a, b):
    a, b = normalize_text(a), normalize_text(b)

    if a == "" and b == "":
        return 1.0
    
    return 1.0 - Levenshtein.distance(a, b) / max(len(a), len(b), 1)

def cell_key(cell):
    return (int(cell["sr"]), int(cell["er"]), int(cell["sc"]), int(cell["ec"]))

def safe_text(c):
    return "" if c is None else normalize_text(c.get("text", ""))

def parse_gt_xml(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.findall("cell"):

        sr = c.get("start_row")
        sc = c.get("start_col")

        if sr is None or sc is None:
            continue

        er = c.get("end_row", sr)
        ec = c.get("end_col", sc)

        text_el = c.find("text")
        text = (text_el.text or "") if text_el is not None else ""

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text(text)
        })

    return out

def coerce_cells(data):

    cells = data.get("cells", data) if isinstance(data, dict) else data
    if not isinstance(cells, list):
        return []
    
    out = []
    for c in cells:
        if not isinstance(c, dict):
            continue

        sr, sc = c.get("sr", c.get("start_row")), c.get("sc", c.get("start_col"))
        if sr is None or sc is None:
            continue

        er, ec = c.get("er", c.get("end_row", sr)), c.get("ec", c.get("end_col", sc))

        try:
            out.append({
                "sr": int(sr), 
                "er": int(er), 
                "sc": int(sc), 
                "ec": int(ec),
                "text": normalize_text(c.get("text", ""))
            })
        except (TypeError, ValueError):
            continue

    return out

def parse_pred_json(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    return coerce_cells(data)

def pred_lookup(pred_cells):

    out = {}
    for c in pred_cells:
        out.setdefault(cell_key(c), c)

    return out

def strict_correct(gt_cell, pred_cell, threshold=LEV_THRESHOLD):

    if pred_cell is None:
        return {"sim": 0.0, "correct": False}
    
    sim = lev_sim(gt_cell["text"], pred_cell["text"])
    
    return {
        "sim": sim, 
        "correct": bool(sim >= threshold)
    }

## Step 2 — Find every table and choose the evaluation protocol

A25 is organized by domain (Biology, CompSci, ICDAR, MatSci) with no pre-defined train/test split, so we pool all tables across domains and split them ourselves by table ID.

In [ ]:

def discover_tables(domains):
    rows = []
    for domain in domains:
        gt_dir  = DATA_ROOT / domain / GT_SUBDIR
        vis_dir = VISION_PRED_ROOT / domain / VISION_PRED_SUBDIR
        txt_dir = TEXT_PRED_ROOT / domain / TEXT_PRED_SUBDIR

        if not gt_dir.exists():
            print(f"[warn] no ground-truth folder for domain '{domain}': {gt_dir}")
            continue

        gt  = {p.stem: p for p in sorted(gt_dir.glob("*.xml"))}
        vis = {p.stem: p for p in vis_dir.glob("*.json")} if vis_dir.exists() else {}
        txt = {p.stem: p for p in txt_dir.glob("*.json")} if txt_dir.exists() else {}

        for s, g in gt.items():
            rows.append({
                "domain": domain, 
                "stem": s, 
                "table_id": f"{domain}::{s}",
                "gt_file": g,
                "vision_file": vis.get(s), 
                "text_file": txt.get(s),
                "vision_missing_file": s not in vis, 
                "text_missing_file": s not in txt,
            })
    return pd.DataFrame(rows)

all_tables_raw = discover_tables(DOMAINS)


def filter_predicted(df):
    before = len(df)
    out = df[~df["vision_missing_file"] & ~df["text_missing_file"]].copy().reset_index(drop=True)

    dropped = before - len(out)

    if dropped:
        print(f"  [filter] dropped {dropped} tables with missing predictions, kept {len(out)}")
        
    return out

all_tables = filter_predicted(all_tables_raw)

print(f"Total tables (with both predictions): {len(all_tables)}")
print(all_tables.groupby("domain").size().rename("tables").to_frame())

## Step 3 — Train / validation / test split (by table)

The split is always **by table**, with the **same seed and procedure as the ML-router
notebook**, so the test tables are identical and the results are directly comparable.

In [ ]:

def split_tables(tables_df, test_size, val_size, seed):

    g1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trv_i, te_i = next(g1.split(tables_df, groups=tables_df["table_id"]))
    trv, te = tables_df.iloc[trv_i].copy(), tables_df.iloc[te_i].copy()

    g2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    tr_i, va_i = next(g2.split(trv, groups=trv["table_id"]))

    return trv.iloc[tr_i].copy(), trv.iloc[va_i].copy(), te

train_tables, val_tables, test_tables = split_tables(
    all_tables, test_size=TEST_SIZE, val_size=VAL_FRACTION, seed=RANDOM_STATE)

print(pd.DataFrame({
    "tables": [len(d) for d in (train_tables, val_tables, test_tables)],
}, index=["train", "val", "test"]))

## Step 4 — Score the single experts per ground-truth cell

For every ground-truth cell we record whether vision was right, whether text was right,
and the oracle outcome (either right). This is the **GT-anchored** protocol: each expert
is looked up at the ground-truth span, exactly how the LLM router's output will be scored.

In [ ]:

def build_cell_dataset(table_df):
    rows = []
    for _, tab in table_df.iterrows():

        gt = parse_gt_xml(tab["gt_file"])
        vl = pred_lookup(parse_pred_json(tab["vision_file"]))
        tl = pred_lookup(parse_pred_json(tab["text_file"]))

        for gi, g in enumerate(gt):

            k = cell_key(g)
            vcell, tcell = vl.get(k), tl.get(k)
            v, t = strict_correct(g, vcell), strict_correct(g, tcell)

            rows.append({
                "domain": tab["domain"], 
                "stem": tab["stem"], 
                "table_id": tab["table_id"],
                "gt_cell_idx": gi, 
                "gt_text": g["text"],
                "vision_text": safe_text(vcell), 
                "text_text": safe_text(tcell),
                "vision_correct": bool(v["correct"]), 
                "text_correct": bool(t["correct"]),
                "oracle_correct": bool(v["correct"] or t["correct"]),
                "one_agent_correct": bool(v["correct"] != t["correct"]),
            })
    return pd.DataFrame(rows)

df_train = build_cell_dataset(train_tables)
df_val = build_cell_dataset(val_tables)
df_test = build_cell_dataset(test_tables)

print(pd.DataFrame({
    "tables": [d["table_id"].nunique() for d in (df_train, df_val, df_test)],
    "cells":  [len(d) for d in (df_train, df_val, df_test)],
}, index=["train", "val", "test"]))

def table_avg(df, col):
    """Accuracy averaged per table, then across tables."""
    return float(df.groupby("table_id")[col].mean().mean()) if len(df) else 0.0


vis_acc = table_avg(df_test, "vision_correct")
txt_acc = table_avg(df_test, "text_correct")
best_single = max(vis_acc, txt_acc)
oracle = table_avg(df_test, "oracle_correct")
oracle_gap = oracle - best_single

print(f"\nVision: {vis_acc*100:.2f}%  |  Text: {txt_acc*100:.2f}%  |  "
      f"Best single: {best_single*100:.2f}%  |  Oracle: {oracle*100:.2f}%  |  "
      f"Oracle gap: {oracle_gap*100:.2f} pp")

## Step 5 — Build the few-shot examples (train tables only)

For in-context learning we want examples that actually *teach the arbitration*: small
tables (so the prompt stays short) where the two experts disagree and the disagreement
is decidable. Candidates are ranked by how often exactly one expert is right, and the
top **N_FEWSHOT** become examples.

Only **train** tables are eligible to find these examples.

In [ ]:

def cells_to_prompt_json(cells):
    return json.dumps(
        {
            "cells": [{
                "sr": c["sr"],
                "er": c["er"], 
                "sc": c["sc"], 
                "ec": c["ec"],
                "text": c["text"]} for c in cells
            ]
        },
        ensure_ascii=False)

train_stats = df_train.groupby("table_id").agg(
    n_cells =("gt_text", "size"),
    disagree =("one_agent_correct", "mean"),
    oracle_acc=("oracle_correct", "mean"),
).reset_index()

candidates = train_stats[(train_stats["n_cells"] <= FEWSHOT_MAX_CELLS) & (train_stats["disagree"] > 0)]
candidates = candidates.sort_values(["disagree", "oracle_acc"], ascending=[False, False])
fewshot_ids = candidates["table_id"].head(N_FEWSHOT).tolist()

tables_by_id = all_tables.set_index("table_id")
fewshot_blocks = []
for i, tid in enumerate(fewshot_ids, 1):

    tab = tables_by_id.loc[tid]

    v_json  = cells_to_prompt_json(parse_pred_json(tab["vision_file"]))
    t_json  = cells_to_prompt_json(parse_pred_json(tab["text_file"]))
    gt_json = cells_to_prompt_json(parse_gt_xml(tab["gt_file"]))

    fewshot_blocks.append(
        f"### EXAMPLE {i}\n"
        f"VISION EXPERT:\n{v_json}\n"
        f"TEXT EXPERT:\n{t_json}\n"
        f"CORRECT FINAL OUTPUT:\n{gt_json}")

print("Few-shot example tables:", fewshot_ids)
print("Total example size:", sum(len(b) for b in fewshot_blocks), "chars")

if fewshot_blocks:
    print("\n" + "\n".join(fewshot_blocks))

## Step 6 — The LLM router prompt and call

The **system prompt** carries the arbitration instructions plus the worked examples.
The **user message** carries the two expert JSONs for the target table. The model must return `{"cells": [...]}` via structured output — the same schema the expertsmthemselves use.

In [ ]:

SYSTEM_PROMPT_CORE = """You are the final arbiter in a dual-expert table extraction system.

Two experts extracted the SAME table:
- the VISION expert read the table image,
- the TEXT expert read the table's text/markup.

You receive both experts' cell lists as JSON and must output the single best final
extraction as JSON: {"cells": [{"sr", "er", "sc", "ec", "text"}, ...]} where sr/er are
the start/end row and sc/ec the start/end column (0-indexed, spans inclusive).

How to arbitrate:
1. Where the experts agree on a cell (same span, same text), keep that cell as-is.
2. Where they give different TEXT for the same position, pick the more plausible value.
   Each expert makes characteristic mistakes: one may leak raw markup (e.g. "\\pm" or
   "pm" instead of the ± sign, "\\mu" instead of the Greek letter mu, "ta_{0.25}" with
   stray braces), the other may over-format or misread characters (e.g. unicode
   subscripts like "ta1.0" written with subscript digits, look-alike characters).
   Prefer the clean, plain, human-readable rendering with real symbols (±, μ, ×) and
   ordinary digits, and with no backslash commands, braces, or "$".
3. Where they disagree on STRUCTURE (different spans, extra or missing cells), prefer
   the expert whose cells form a consistent rectangular grid without overlaps.
4. Never invent a cell that appears in neither expert. Cover the whole table exactly once.
5. Output ONLY the JSON object, no commentary."""

SYSTEM_PROMPT = (SYSTEM_PROMPT_CORE
                 + "\n\n## WORKED EXAMPLES — learn the arbitration patterns from these\n\n"
                 + "\n\n".join(fewshot_blocks))

CACHE_DIR = CACHE_ROOT / f"few-shot-{hashlib.md5(SYSTEM_PROMPT.encode('utf-8')).hexdigest()[:8]}"

def llm_router_call(user_prompt):
    kwargs = dict(
        model=ROUTER_MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "cell-info", "schema": CELL_SCHEMA},
        },
        seed=RANDOM_STATE,
        max_completion_tokens=32768,
        stream=True,
    )

    last_err = None
    for attempt in range(MAX_RETRIES + 1):

        try:
            stream = client.chat.completions.create(**kwargs)
            parts = []

            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
                    parts.append(chunk.choices[0].delta.content)

            return "".join(parts)
        
        except Exception as e:

            last_err = e
            time.sleep(RETRY_BACKOFF_SEC * (attempt + 1))

    raise last_err

THINK_RE = re.compile(r"<think>.*?</think>", re.S)

def extract_cells_from_response(raw):
    """Parse the router's reply into cells; anything unparseable scores as empty (all wrong)."""
    if raw is None:
        return []
    
    s = THINK_RE.sub(" ", str(raw)).strip()
    if not s:
        return []
    
    dec, best, i = json.JSONDecoder(), None, 0

    while True:
        j = s.find("{", i)

        if j == -1:
            break

        try:
            obj, end = dec.raw_decode(s, j)

            if isinstance(obj, (dict, list)) and (best is None or obj):
                best = obj

            i = end
        except Exception:
            i = j + 1

    return coerce_cells(best) if best is not None else []

def run_router_on_table(tab):
    """Returns (router_cells, from_cache). Calls the LLM only on a cache miss."""

    cache = CACHE_DIR / tab["domain"] / f"{tab['stem']}.json"
    if cache.exists():
        raw = json.loads(cache.read_text(encoding="utf-8"))["raw"]

        return extract_cells_from_response(raw), True
    
    v_json = cells_to_prompt_json(parse_pred_json(tab["vision_file"]))
    t_json = cells_to_prompt_json(parse_pred_json(tab["text_file"]))

    user_prompt = (f"VISION EXPERT:\n{v_json}\n\n"
                   f"TEXT EXPERT:\n{t_json}\n\n"
                   "Produce the final cells JSON for this table.")
    
    raw = llm_router_call(user_prompt)
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps({"model": ROUTER_MODEL_NAME, "table_id": tab["table_id"], "raw": raw}, ensure_ascii=False), encoding="utf-8")
    
    return extract_cells_from_response(raw), False

print(f"System prompt: {len(SYSTEM_PROMPT):,} chars  ->  cache {CACHE_DIR}")

## Step 7 — Run the LLM router on every test table

In [ ]:
# --- Run the router over the test tables (cached, resumable) -----------------
rows, failures = [], 0

for _, tab in tqdm(test_tables.iterrows(), total=len(test_tables), desc="router"):

    try:

        router_cells, from_cache = run_router_on_table(tab)
    except Exception as e:

        print(f"  [fail] {tab['table_id']}: {type(e).__name__}: {e}")
        router_cells, from_cache = [], True
        failures += 1

    rl = pred_lookup(router_cells)
    for gi, g in enumerate(parse_gt_xml(tab["gt_file"])):

        rcell = rl.get(cell_key(g))
        rows.append({
            "domain": tab["domain"], 
            "table_id": tab["table_id"],
            "gt_cell_idx": gi, 
            "router_text": safe_text(rcell),
            "router_correct": bool(strict_correct(g, rcell)["correct"]),
        })

    if not from_cache:
        time.sleep(SLEEP_BETWEEN_CALLS_SEC)

router_df  = pd.DataFrame(rows)
router_acc = table_avg(router_df, "router_correct")
print(f"Router acc {router_acc*100:.2f}%  ({failures} failed calls)")

## Comparison 2 — LLM router vs single experts and the oracle

The same summary as the ML-router notebook: single-expert accuracies, the router's
accuracy, the oracle ceiling, the router's gain over the best single expert, and how
much of the oracle gap it closes.

In [ ]:

r_gain = router_acc - best_single
table1 = pd.DataFrame([{
    "set": "test",
    "router model": ROUTER_MODEL_NAME,
    "averaging": "table-average",
    "tables": int(df_test["table_id"].nunique()),
    "cells":  int(len(df_test)),
    "Vision acc %": round(vis_acc * 100, 2),
    "Text acc %": round(txt_acc * 100, 2),
    "Best single %": round(best_single * 100, 2),
    "Router acc %": round(router_acc * 100, 2),
    "Oracle acc %": round(oracle * 100, 2),
    "Gain over best single pp": round(r_gain * 100, 2),
    "Oracle gap pp": round(oracle_gap * 100, 2),
    "Gap closed %": round(r_gain / oracle_gap * 100, 2) if oracle_gap > 0 else 0.0,
}])

display(table1)